In [1]:
!pip install mlflow


In [2]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# 1. Create simulated Loan Dataset
np.random.seed(42)
n_samples = 200
data = {
    "Income": np.random.randint(20000, 120000, n_samples),
    "Credit_Score": np.random.randint(300, 850, n_samples),
    "Age": np.random.randint(21, 65, n_samples),
    "Loan_Amount": np.random.randint(5000, 50000, n_samples),
}
df = pd.DataFrame(data)
df["Loan_Status"] = (
    (df["Credit_Score"] > 600) & (df["Income"] > df["Loan_Amount"] * 1.5)
).astype(int)

# 2. Split Features and Target
X = df[["Income", "Credit_Score", "Age", "Loan_Amount"]]
y = df["Loan_Status"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Initialize MLflow Experiment
mlflow.set_experiment("Loan_Approval_Tracking")

# 4. Start MLflow Run and log parameters, metrics, and models
with mlflow.start_run():
    n_estimators = 100
    max_depth = 5
    random_state = 42

    # Train
    rf_model = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth, random_state=random_state
    )
    rf_model.fit(X_train, y_train)

    # Log Parameters
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", random_state)

    # Evaluate & Log Metrics
    y_pred = rf_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    # Save Model Artifact
    mlflow.sklearn.log_model(rf_model, artifact_path="loan_rf_model")

    print(f"✅ Execution Complete! Test Accuracy: {accuracy * 100:.2f}%")


2026/08/22 04:27:55 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/22 04:27:55 INFO mlflow.store.db.utils: Updating database tables
2026/08/22 04:28:01 INFO mlflow.tracking.fluent: Experiment with name 'Loan_Approval_Tracking' does not exist. Creating a new experiment.
2026/08/22 04:28:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Execution Complete! Test Accuracy: 100.00%
